[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ThanhDatVN/AI_journey/blob/main/notebooks/13_svm_and_kernels/04_svm_in_practice_and_failure_cases.ipynb)

# Bài 04 — SVM trong thực tế và các failure case

**🎯 Mục tiêu**
- Quy trình chuẩn: **scaling bắt buộc** → RBF → grid search $C$, $\gamma$ trên thang log.
- Xác suất từ SVM: **Platt scaling** (`probability=True`) và chi phí của nó.
- **SVR**: hồi quy với $\varepsilon$-insensitive loss.
- Failure case: dữ liệu lớn ($O(n^2)$–$O(n^3)$), không scale, mất cân bằng, nhiều feature nhiễu; khi nào dùng `LinearSVC`/`SGDClassifier`.
- Hoàn thiện template model cho SVM.

In [ ]:
# ⚙️ Setup — chạy được cả local lẫn Google Colab (local: đã `pip install -e .` thì bỏ qua)
import sys
if "google.colab" in sys.modules:
    !test -d /content/ml-course || git clone -q --depth 1 https://github.com/ThanhDatVN/AI_journey /content/ml-course
    !pip install -q -e /content/ml-course
    sys.path.insert(0, "/content/ml-course/src")
    %cd /content/ml-course/notebooks/13_svm_and_kernels

In [ ]:
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.model_selection import GridSearchCV, cross_val_score, train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC, SVR, LinearSVC

from ml_course.datasets import load_digits_data, make_classification_2d
from ml_course.visualization import set_style

set_style()

## 1. Scaling: không có là hỏng

RBF kernel dùng **khoảng cách** — giống KNN (chương 08), feature có thang đo lớn sẽ áp đảo:

In [ ]:
rng = np.random.default_rng(0)
Xm, ym = make_classification_2d("moons", n=600, noise=0.25, seed=3)
X_bad = Xm * np.array([1.0, 1000.0])                          # feature 2 đổi đơn vị
for name, model in [("không scale", SVC(kernel="rbf")), ("StandardScaler + SVC", make_pipeline(StandardScaler(), SVC(kernel="rbf")))]:
    print(f"{name:22s} accuracy (5-fold CV) = {cross_val_score(model, X_bad, ym, cv=5).mean():.3f}")

## 2. Chọn C và γ: grid search trên thang log

In [ ]:
X, y, _ = load_digits_data()
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, random_state=0, stratify=y)
pipe = make_pipeline(StandardScaler(), SVC(kernel="rbf"))
param_grid = {"svc__C": np.logspace(-1, 3, 5), "svc__gamma": np.logspace(-4, 0, 5)}
grid = GridSearchCV(pipe, param_grid, cv=4).fit(X_tr, y_tr)
scores = grid.cv_results_["mean_test_score"].reshape(len(param_grid["svc__C"]), len(param_grid["svc__gamma"]))
plt.imshow(scores, cmap="viridis")
plt.xticks(range(5), [f"{g:.0e}" for g in param_grid["svc__gamma"]])
plt.yticks(range(5), [f"{c:g}" for c in param_grid["svc__C"]])
plt.xlabel("gamma")
plt.ylabel("C")
for i in range(5):
    for j in range(5):
        plt.text(j, i, f"{scores[i, j]:.3f}", ha="center", va="center", color="w", fontsize=8)
plt.colorbar(label="accuracy (CV)")
plt.title("Digits: vùng tốt là một 'dải chéo' — C và γ bù trừ nhau")
plt.show()
print("tốt nhất:", grid.best_params_, f"| test accuracy = {grid.score(X_te, y_te):.4f}")

Mẹo: mặc định `gamma="scale"` = $1/(d\cdot\operatorname{Var}(X))$ thường là điểm khởi đầu tốt; tìm $C$ trong $[10^{-2}, 10^3]$, $\gamma$ quanh giá trị mặc định × $[10^{-2}, 10^2]$.

## 3. Xác suất: Platt scaling

SVM cho điểm số $f(\mathbf x)$, không phải xác suất. `probability=True` fit thêm một logistic regression $P(y = 1 \mid f) = \sigma(Af + B)$ bằng **5-fold CV nội bộ** → train **chậm hơn nhiều** và `predict_proba` có thể **không nhất quán** với `predict`:

In [ ]:
Xb, yb = make_classification_2d("moons", n=3000, noise=0.3, seed=5)
for prob in [False, True]:
    start = time.perf_counter()
    m = SVC(kernel="rbf", probability=prob, random_state=0).fit(Xb, yb)
    print(f"probability={prob!s:5s}: train {time.perf_counter() - start:.2f}s")
m = SVC(kernel="rbf", probability=True, random_state=0).fit(Xb, yb)
inconsistent = np.mean((m.predict_proba(Xb)[:, 1] >= 0.5).astype(int) != m.predict(Xb))
print(f"tỉ lệ điểm mà argmax(predict_proba) ≠ predict: {inconsistent:.2%}")

Khuyến nghị: dùng `decision_function` để xếp hạng (ROC-AUC), và nếu cần xác suất thì `CalibratedClassifierCV` một cách tường minh (chương 37).

## 4. SVR — hồi quy với ε-insensitive loss

Bỏ qua sai số nhỏ hơn $\varepsilon$ ("ống" quanh hàm dự đoán); chỉ điểm **ngoài ống** là support vectors:

$$\min \frac12\lVert\mathbf w\rVert^2 + C\sum_i \max(0, |y_i - f(\mathbf x_i)| - \varepsilon)$$

In [ ]:
from ml_course.datasets import make_regression_1d

Xr, yr = make_regression_1d(n=150, kind="sin", noise=0.3, seed=0)
grid_x = np.linspace(Xr.min(), Xr.max(), 300).reshape(-1, 1)
fig, axes = plt.subplots(1, 3, figsize=(18, 4.2))
for ax, eps in zip(axes, [0.1, 0.5, 1.5]):
    svr = SVR(kernel="rbf", C=10, epsilon=eps, gamma=1.0).fit(Xr, yr)
    pred = svr.predict(grid_x)
    ax.scatter(Xr, yr, s=10, alpha=0.5)
    ax.plot(grid_x, pred, "r")
    ax.fill_between(grid_x.ravel(), pred - eps, pred + eps, color="r", alpha=0.15)
    ax.scatter(Xr[svr.support_], yr[svr.support_], s=30, facecolors="none", edgecolors="k", lw=0.6)
    ax.set_title(f"ε = {eps}: {len(svr.support_)}/{len(Xr)} support vectors")
plt.show()

## 5. Failure case: dữ liệu lớn

Kernel SVM cần ma trận kernel $n \times n$ và thời gian train khoảng $O(n^2)$–$O(n^3)$ → không khả thi với hàng trăm nghìn mẫu. Dự đoán cũng tốn $O(n_{SV}\cdot d)$.

In [ ]:
sizes = [1000, 2000, 4000, 8000]
times = {"SVC (RBF)": [], "LinearSVC": []}
for n in sizes:
    Xs_, ys_ = make_classification_2d("moons", n=n, noise=0.3, seed=1)
    Xs_ = np.hstack([Xs_, rng.normal(size=(n, 18))])
    for name, model in [("SVC (RBF)", SVC(kernel="rbf")), ("LinearSVC", LinearSVC())]:
        start = time.perf_counter()
        model.fit(Xs_, ys_)
        times[name].append(time.perf_counter() - start)
for name, t in times.items():
    plt.plot(sizes, t, "o-", label=name)
plt.xlabel("số mẫu")
plt.ylabel("thời gian train (s)")
plt.legend()
plt.title("Kernel SVM tăng siêu tuyến tính theo n")
plt.show()
for i, n in enumerate(sizes):
    print(f"n = {n}: SVC {times['SVC (RBF)'][i]:.2f}s | LinearSVC {times['LinearSVC'][i]:.3f}s")

**Khắc phục:** `LinearSVC` / `SGDClassifier(loss="hinge")` cho dữ liệu lớn (đặc biệt văn bản nhiều chiều), xấp xỉ kernel (`Nystroem`, `RBFSampler` — random Fourier features) + model tuyến tính, hoặc chuyển sang gradient boosting.

In [ ]:
from sklearn.kernel_approximation import Nystroem
from sklearn.linear_model import SGDClassifier

Xl, yl = make_classification_2d("moons", n=30000, noise=0.3, seed=7)
Xl_tr, Xl_te, yl_tr, yl_te = train_test_split(Xl, yl, test_size=0.3, random_state=0)
for name, model in [("LinearSVC (tuyến tính)", LinearSVC()),
                    ("Nystroem(300) + SGD hinge", make_pipeline(StandardScaler(), Nystroem(gamma=1.0, n_components=300, random_state=0),
                                                                 SGDClassifier(loss="hinge", random_state=0)))]:
    start = time.perf_counter()
    model.fit(Xl_tr, yl_tr)
    print(f"{name:26s} accuracy test = {model.score(Xl_te, yl_te):.3f} | train {time.perf_counter() - start:.2f}s")

## 6. Failure case: mất cân bằng lớp và feature nhiễu

- **Mất cân bằng:** hinge loss đếm mọi vi phạm như nhau → ranh giới bị đẩy về phía lớp thiểu số. Dùng `class_weight="balanced"` (nhân $C$ theo lớp) và đánh giá bằng metric phù hợp.
- **Nhiều feature nhiễu:** RBF dựa trên khoảng cách → giống KNN, bị "pha loãng" bởi feature vô ích (chương 08, bài 04).

In [ ]:
from sklearn.metrics import balanced_accuracy_score, recall_score

Xi, yi = make_classification_2d("blobs", n=4000, noise=0.8, seed=9)
minority = np.flatnonzero(yi == 1)
drop = rng.choice(minority, int(0.9 * len(minority)), replace=False)
keep = np.setdiff1d(np.arange(len(yi)), drop)
Xi, yi = Xi[keep], yi[keep]
Xi_tr, Xi_te, yi_tr, yi_te = train_test_split(Xi, yi, test_size=0.3, random_state=0, stratify=yi)
print(f"tỉ lệ lớp thiểu số: {yi.mean():.1%}")
for cw in [None, "balanced"]:
    m = SVC(kernel="rbf", class_weight=cw).fit(Xi_tr, yi_tr)
    print(f"class_weight={cw!s:9s}: recall lớp thiểu số = {recall_score(yi_te, m.predict(Xi_te)):.3f} | "
          f"balanced accuracy = {balanced_accuracy_score(yi_te, m.predict(Xi_te)):.3f}")

## 7. Template model: SVM

| Phần | Support Vector Machine |
|---|---|
| Problem | phân loại (nhị phân; nhiều lớp qua OvO/OvR), hồi quy (SVR), phát hiện bất thường (One-Class SVM) |
| Intuition | ranh giới có **lề lớn nhất**; chỉ support vectors quan trọng |
| Geometry | siêu phẳng trong không gian kernel → phi tuyến trong không gian gốc |
| Probability | không có sẵn — Platt scaling / calibration |
| Objective | $\frac12\lVert\mathbf w\rVert^2 + C\sum\text{hinge}$ (primal) ⇔ QP đối ngẫu với kernel |
| Optimization | QP lồi: SMO (libsvm), coordinate descent (liblinear), Pegasos/SGD |
| Assumptions | độ đo tương tự (kernel) có ý nghĩa; feature đã scale |
| Complexity | kernel: train $O(n^2)$–$O(n^3)$, bộ nhớ $O(n^2)$; dự đoán $O(n_{SV} d)$. Linear: $O(nd)$ |
| Hyperparameters | `C`, `kernel`, `gamma`, `degree`, `epsilon` (SVR), `class_weight` |
| Preprocessing | **bắt buộc scaling**; encode categorical; không tự xử lý missing |
| Strengths | tốt với dữ liệu vừa & nhiều chiều (văn bản, ảnh nhỏ), lồi (nghiệm toàn cục), thưa, kernel linh hoạt |
| Weaknesses | chậm khi $n$ lớn, nhạy scaling & $C/\gamma$, không có xác suất, khó diễn giải với kernel |
| Interpretation | linear: hệ số $\mathbf w$; kernel: support vectors, permutation importance |

## 🧠 Kiểm tra nhanh

1. Vì sao scaling bắt buộc với RBF SVM?
2. `probability=True` làm gì và vì sao train chậm hơn?
3. Bạn có 2 triệu văn bản, 500 000 feature TF-IDF: SVC(rbf) hay LinearSVC? Vì sao?
4. SVR với $\varepsilon$ lớn hơn thì số support vectors thay đổi thế nào?

## 📌 Tóm tắt

- Pipeline: scale → RBF → grid search $C$, $\gamma$ (log) → nếu cần xác suất thì calibrate.
- Kernel SVM mạnh cho dữ liệu cỡ vừa; dữ liệu lớn → linear SVM / xấp xỉ kernel / boosting.
- `class_weight` cho mất cân bằng; feature nhiễu và không scale làm hỏng RBF.

## 📚 Tài liệu tham khảo

- Hsu, Chang & Lin — *A Practical Guide to Support Vector Classification* (LIBSVM)
- Rahimi & Recht (2007) — *Random Features for Large-Scale Kernel Machines*
- scikit-learn User Guide — Support Vector Machines